In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_frmri_articulatory_features

In [15]:
df = pd.read_csv("../dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[1])

In [16]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.717,2.717,2.907,f,i,f,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
1,2.717,2.907,2.907,2.937,i,l,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
2,2.907,2.937,2.937,3.067,l,u,f_i_l,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
3,3.097,3.157,3.157,3.237,l,a,l,la,4,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
4,3.237,3.317,3.317,3.377,f,ʁ,f,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
...,...,...,...,...,...,...,...,...,...,...,...,...
22432,33.071,33.131,33.131,33.211,ʁ,e,p_ʁ,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22433,33.131,33.211,33.211,33.391,e,s,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22434,33.211,33.391,33.391,33.671,s,i,p_ʁ_e_s,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22435,33.391,33.671,33.671,33.801,i,z,p_ʁ_e_s_i,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(136, 136),
    latent_dim=64,
    input_channels=1,
    output_activation=None,
    stn_mode="shared",
    stn_transform="rigid",
).to(device)

ckpt = torch.load(
    "../../encoder_training/ckpts/frmri/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()


ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/frmri/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [18]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.717,2.717,2.907,f,i,f,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
1,2.717,2.907,2.907,2.937,i,l,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
2,2.907,2.937,2.937,3.067,l,u,f_i_l,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
3,3.097,3.157,3.157,3.237,l,a,l,la,4,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
4,3.237,3.317,3.317,3.377,f,ʁ,f,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
...,...,...,...,...,...,...,...,...,...,...,...,...
22432,33.071,33.131,33.131,33.211,ʁ,e,p_ʁ,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22433,33.131,33.211,33.211,33.391,e,s,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22434,33.211,33.391,33.391,33.671,s,i,p_ʁ_e_s,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22435,33.391,33.671,33.671,33.801,i,z,p_ʁ_e_s_i,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4


In [19]:
import cv2
with open("../../encoder_training/scalers/frmri_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../../encoder_training/scalers/frmri_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [20]:
def get_vectors(art_f, ac_f, start, end, subject):
    stats = recording_stats(art_f)
    articulatory_features = get_frmri_articulatory_features(
        art_f,
        start,
        end,
        stats=stats
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )
        if articulatory_features.shape[-2:]==(136, 136):
            art_out, art_z = art_model(
                articulatory_features,
                return_latent=True
            )
        else:
            art_out, art_z = None, None
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = apply_scaler(mfccs, ac_scalers[subject])
        mfccs = (
            torch.from_numpy(mfccs)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs, return_latent=True)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [21]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [22]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    # "art_raw_dist": [],
    "ac_latent_dist": [],
    # "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    # art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    # ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    # dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    # dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_frmri.csv", index=False)

100%|██████████| 22437/22437 [1:45:20<00:00,  3.55it/s]  
